In [1]:
pip install numpy pandas networkx

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: C:\Users\User\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
import networkx as nx
from math import radians, sin, cos, sqrt, atan2
import json, warnings
warnings.filterwarnings("ignore")

In [4]:

 
zone_df      = pd.read_csv( "zone_data_augmented_aug2.csv")
priority_df  = pd.read_csv( "zone_priority_augmented_aug2.csv")
demand_df    = pd.read_csv( "zone_demand_summary_augmented_aug2.csv")
schedule_df  = pd.read_csv( "charging_schedule_augmented_aug2.csv")
gaps_df      = pd.read_csv( "coverage_gaps_augmented_aug2.csv")

In [5]:
zone_agg  = zone_df.groupby("zone").mean().reset_index()
prio_agg  = priority_df.groupby("zone")[["population","stations","ev_density","score"]].mean().reset_index()
dem_agg   = demand_df.groupby("zone").mean().reset_index()
 

peak_stats = schedule_df.groupby("zone")["demand"].agg(
    avg_demand="mean", max_demand="max", demand_std="std"
).reset_index()
 
peak_hour = (
    schedule_df[schedule_df["recommendation"] == "PEAK"]
    .groupby("zone")["hour"].mean()
    .reset_index()
    .rename(columns={"hour": "avg_peak_hour"})
)
 

merge = (zone_agg                                    
    .merge(prio_agg[["zone","score"]], on="zone")      
    .merge(dem_agg[["zone","avg_demand"]], on="zone")  
    .merge(peak_stats, on="zone")                    
    .merge(peak_hour, on="zone", how="left")
)

if "avg_demand_x" in merge.columns:
    merge = merge.rename(columns={"avg_demand_x": "avg_demand"}).drop(columns=["avg_demand_y"], errors="ignore")
 

merge["congestion_index"] = (
    (merge["ev_density"] * merge["max_demand"]) /
    (merge["stations"].clip(lower=0.5))
)

ci_median = merge["congestion_index"].median()
merge["overloaded"] = merge["congestion_index"] > ci_median
 
print("=" * 60)
print("ZONE SUMMARY")
print("=" * 60)
print(merge[["zone","lat","lon","population","stations","ev_density",
             "avg_demand","max_demand","congestion_index","overloaded"]].to_string(index=False))
print()

ZONE SUMMARY
zone       lat       lon  population  stations  ev_density  avg_demand  max_demand  congestion_index  overloaded
   A 13.032671 76.976975 4990.402919  2.003917  250.962070  142.285530  392.833003      49196.735714       False
   B 13.018140 77.020853 8063.239640  1.001073  400.219604  153.766712  451.206516     180388.057643        True
   C 13.047191 78.310263 3009.781765  0.000000  150.166444  133.258180  323.426171      97135.516161       False
   D 13.015658 78.157127 9981.031543  3.002453  496.498740  144.152540  396.813999      65618.891375       False
   E 12.990161 77.273521 6044.050890  1.004410  300.450487  144.093663  411.561404     123110.958276        True



In [6]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    phi1, phi2 = radians(lat1), radians(lat2)
    dphi  = radians(lat2 - lat1)
    dlam  = radians(lon2 - lon1)
    a = sin(dphi/2)**2 + cos(phi1)*cos(phi2)*sin(dlam/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))


In [7]:
def minmax_norm(series):
    mn, mx = series.min(), series.max()
    return (series - mn) / (mx - mn + 1e-9)
 
merge["ci_norm"]  = minmax_norm(merge["congestion_index"])
merge["dem_norm"] = minmax_norm(merge["avg_demand"])
 
def edge_weight(row_u, row_v):
    dist = haversine_km(row_u.lat, row_u.lon, row_v.lat, row_v.lon)
    congestion_penalty = (row_u.ci_norm + row_v.ci_norm) / 2.0
    demand_pressure    = (row_u.dem_norm + row_v.dem_norm) / 2.0
    w = dist * (1 + congestion_penalty) * (1 + demand_pressure)
    return round(w, 4)

In [8]:
G = nx.Graph()
zones = merge["zone"].tolist()
 
for _, row in merge.iterrows():
    G.add_node(row.zone,
               lat=row.lat, lon=row.lon,
               population=row.population,
               stations=row.stations,
               ev_density=row.ev_density,
               avg_demand=row.avg_demand,
               max_demand=row.max_demand,
               congestion_index=row.congestion_index,
               overloaded=bool(row.overloaded),
               ci_norm=row.ci_norm,
               dem_norm=row.dem_norm)

for i, z1 in enumerate(zones):
    for j, z2 in enumerate(zones):
        if i >= j:
            continue
        r1 = merge[merge.zone == z1].iloc[0]
        r2 = merge[merge.zone == z2].iloc[0]
        dist_km = haversine_km(r1.lat, r1.lon, r2.lat, r2.lon)
        w       = edge_weight(r1, r2)
        G.add_edge(z1, z2,
                   distance_km=round(dist_km, 3),
                   weight=w)
 
print("=" * 60)
print("GRAPH EDGES (raw distance & composite weight)")
print("=" * 60)
print(f"{'Edge':<8} {'Distance (km)':>15} {'Composite w':>14}")
print("-" * 40)
for u, v, d in sorted(G.edges(data=True), key=lambda x: x[2]["distance_km"]):
    print(f"{u}↔{v}   {d['distance_km']:>12.2f} km   {d['weight']:>10.4f}")
print()

GRAPH EDGES (raw distance & composite weight)
Edge       Distance (km)    Composite w
----------------------------------------
A↔B           5.02 km      12.9538
C↔D          16.96 km      26.7233
B↔E          27.55 km      86.5994
A↔E          32.47 km      61.7773
D↔E          95.78 km     196.9583
C↔E         112.50 km     208.2599
B↔D         123.10 km     339.6265
A↔D         127.86 km     201.8580
B↔C         139.72 km     352.6612
A↔C         144.44 km     208.4285



In [11]:

print("FLOYD-WARSHALL: All-Pairs Shortest Paths (composite weight)")

 

fw_dist = dict(nx.floyd_warshall(G, weight="weight"))
fw_pred = dict(nx.floyd_warshall_predecessor_and_distance(G, weight="weight")[0])
 
def reconstruct_path(pred, u, v):
   
    if u == v:
        return [u]
    path = [v]
    while path[-1] != u:
        path.append(pred[u][path[-1]])
    path.reverse()
    return path
 
print(f"\n{'From→To':<10}", end="")
for z in zones:
    print(f"{z:>10}", end="")
print()
print("-" * (10 + 10*len(zones)))
for u in zones:
    print(f"{u:<10}", end="")
    for v in zones:
        val = fw_dist[u][v]
        print(f"{val:>10.3f}", end="")
    print()
print()


FLOYD-WARSHALL: All-Pairs Shortest Paths (composite weight)

From→To            A         B         C         D         E
------------------------------------------------------------
A              0.000    12.954   208.429   201.858    61.777
B             12.954     0.000   221.382   214.812    74.731
C            208.429   221.382     0.000    26.723   208.260
D            201.858   214.812    26.723     0.000   196.958
E             61.777    74.731   208.260   196.958     0.000



In [12]:
 
DIST_THRESHOLD_KM = 50.0  
 

print(f"COVERAGE GAP ANALYSIS  (threshold = {DIST_THRESHOLD_KM} km)")

 
merge["ev_dens_norm"] = minmax_norm(merge["ev_density"])
merge["sta_norm"]     = minmax_norm(merge["stations"])
 
gap_results = []
 
for i, u in enumerate(zones):
    for j, v in enumerate(zones):
        if i >= j:
            continue
 
        raw_dist = G[u][v]["distance_km"]
        direct_w = G[u][v]["weight"]
 
        if raw_dist < DIST_THRESHOLD_KM:
            continue
 
     
        candidates = []
        for k in zones:
            if k in (u, v):
                continue
            w_uk = fw_dist[u][k]
            w_kv = fw_dist[k][v]
            routed_cost = w_uk + w_kv
 
         
            cost_reduction = direct_w - routed_cost
 
            rk   = merge[merge.zone == k].iloc[0]
            benefit = (
                (1 + rk.ev_dens_norm)
                * (1 + (1 - rk.sta_norm))
                * (1 + rk.dem_norm)
            )
            final_score = max(cost_reduction, 0) * benefit
 
            candidates.append({
                "k":              k,
                "routed_cost":    round(routed_cost, 4),
                "cost_reduction": round(cost_reduction, 4),
                "ev_density_k":   round(rk.ev_density, 1),
                "stations_k":     round(rk.stations, 2),
                "congestion_k":   round(rk.congestion_index, 1),
                "benefit_score":  round(final_score, 4),
            })
 
        if not candidates:
            continue
 
        best = max(candidates, key=lambda x: x["benefit_score"])
        full_path = reconstruct_path(fw_pred, u, v)
 
        gap_results.append({
            "gap_pair":          f"{u}↔{v}",
            "raw_distance_km":   round(raw_dist, 2),
            "direct_weight":     round(direct_w, 4),
            "recommended_k":     best["k"],
            "routed_cost":       best["routed_cost"],
            "cost_reduction_%":  round(100 * best["cost_reduction"] / (direct_w + 1e-9), 1),
            "benefit_score":     best["benefit_score"],
            "all_candidates":    sorted(candidates, key=lambda x: -x["benefit_score"]),
            "fw_path":           full_path,
        })
 
if gap_results:
    for g in gap_results:
        print(f"\nGAP: {g['gap_pair']}")
        print(f"  Raw distance      : {g['raw_distance_km']} km")
        print(f"  Direct weight     : {g['direct_weight']}")
        print(f"  Floyd-Warshall path: {' → '.join(g['fw_path'])}")
        print(f"   Best intermediate: Zone {g['recommended_k']}")
        print(f"    Routed cost      : {g['routed_cost']}  (direct was {g['direct_weight']})")
        print(f"    Cost reduction   : {g['cost_reduction_%']}%")
        print(f"    Benefit score    : {g['benefit_score']}")
        print(f"  All candidates ranked:")
        for c in g["all_candidates"]:
            print(f"    Zone {c['k']}: benefit={c['benefit_score']:.4f} "
                  f"| EV density={c['ev_density_k']} "
                  f"| stations={c['stations_k']:.1f} "
                  f"| congestion={c['congestion_k']:.1f}")
else:
    print("No gaps exceed threshold. All zones are within coverage range.")

COVERAGE GAP ANALYSIS  (threshold = 50.0 km)

GAP: A↔C
  Raw distance      : 144.44 km
  Direct weight     : 208.4285
  Floyd-Warshall path: A → C
   Best intermediate: Zone B
    Routed cost      : 234.3361  (direct was 208.4285)
    Cost reduction   : -12.4%
    Benefit score    : 0.0
  All candidates ranked:
    Zone B: benefit=0.0000 | EV density=400.2 | stations=1.0 | congestion=180388.1
    Zone D: benefit=0.0000 | EV density=496.5 | stations=3.0 | congestion=65618.9
    Zone E: benefit=0.0000 | EV density=300.5 | stations=1.0 | congestion=123111.0

GAP: A↔D
  Raw distance      : 127.86 km
  Direct weight     : 201.858
  Floyd-Warshall path: A → D
   Best intermediate: Zone B
    Routed cost      : 227.7656  (direct was 201.858)
    Cost reduction   : -12.8%
    Benefit score    : 0.0
  All candidates ranked:
    Zone B: benefit=0.0000 | EV density=400.2 | stations=1.0 | congestion=180388.1
    Zone C: benefit=0.0000 | EV density=150.2 | stations=0.0 | congestion=97135.5
    Zone

In [13]:
print()
print("=" * 60)
print("NEW STATION BUILD PRIORITY RANKING")
print("=" * 60)
 
gap_zones = {g["recommended_k"] for g in gap_results}
 
build_scores = []
for _, row in merge.iterrows():
    z = row.zone
    gap_bonus = 2.0 if z in gap_zones else 1.0  
 
    raw_score = (
          0.35 * row.ev_dens_norm
        + 0.25 * minmax_norm(merge["population"])[merge[merge.zone==z].index[0]]
        + 0.20 * row.ci_norm
        + 0.15 * row.dem_norm
        - 0.15 * row.sta_norm
    ) * gap_bonus
 
    stations_existing = round(row.stations, 1)
    stations_recommended = max(1, round(row.ev_density / 150))  
 
    build_scores.append({
        "zone":                 z,
        "ev_density":           round(row.ev_density, 1),
        "population":           round(row.population),
        "existing_stations":    round(row.stations, 2),
        "congestion_index":     round(row.congestion_index, 1),
        "overloaded":           row.overloaded,
        "graph_recommended":    z in gap_zones,
        "build_score":          round(raw_score, 4),
        "stations_recommended": stations_recommended,
    })
 
build_df = pd.DataFrame(build_scores).sort_values("build_score", ascending=False).reset_index(drop=True)
build_df["position"] = build_df.index + 1
 
print(f"\n{'Rank':<6}{'Zone':<6}{'Score':>8}{'EV Dens':>10}{'Popul':>10}"
      f"{'Exist.Sta':>11}{'Recomm.Sta':>12}{'Overloaded':>12}{'GraphRec':>10}")
print("-" * 85)
for _, r in build_df.iterrows():
    ovl  = "YES " if r.overloaded        else "no"
    grec = " YES"  if r.graph_recommended else "-"
    print(f"{int(r.position):<6}{r.zone:<6}{r.build_score:>8.4f}{r.ev_density:>10.1f}"
          f"{int(r.population):>10}{r.existing_stations:>11.1f}"
          f"{r.stations_recommended:>12}{ovl:>12}{grec:>10}")



NEW STATION BUILD PRIORITY RANKING

Rank  Zone     Score   EV Dens     Popul  Exist.Sta  Recomm.Sta  Overloaded  GraphRec
-------------------------------------------------------------------------------------
1     B       1.4678     400.2      8063        1.0           3        YES        YES
2     D       1.1094     496.5      9981        3.0           3          no       YES
3     E       0.4024     300.5      6044        1.0           2        YES          -
4     A       0.2776     251.0      4990        2.0           2          no       YES
5     C       0.0731     150.2      3010        0.0           1          no         -


In [15]:


print("SMART REROUTING: If Zone X is overloaded → go to Zone Y")

 
for u in zones:
    u_data = G.nodes[u]
    if not u_data["overloaded"]:
        continue
    alternatives = []
    for v in zones:
        if v == u:
            continue
        v_data = G.nodes[v]
        cost = fw_dist[u][v]

        penalty = 1.5 if v_data["overloaded"] else 1.0
        effective = cost * penalty
        path = reconstruct_path(fw_pred, u, v)
        alternatives.append({
            "dest": v,
            "fw_cost": round(cost, 3),
            "dest_overloaded": v_data["overloaded"],
            "effective_cost": round(effective, 3),
            "path": " → ".join(path),
        })
    alternatives.sort(key=lambda x: x["effective_cost"])
    best_alt = alternatives[0]
    print(f"\n  Zone {u} (OVERLOADED) → best reroute: Zone {best_alt['dest']}")
    print(f"    Path       : {best_alt['path']}")
    print(f"    FW cost    : {best_alt['fw_cost']}")
    print(f"    Dest loaded: {'Yes' if best_alt['dest_overloaded'] else 'No'}")
    print(f"  All options  :")
    for a in alternatives:
        tag = " [overloaded]" if a["dest_overloaded"] else ""
        print(f"    → {a['dest']}  path={a['path']}  eff_cost={a['effective_cost']}{tag}")
 
 

 

print("EXPLAINABILITY SUMMARY — WHY EACH ZONE RANKED WHERE IT DID")

for _, r in build_df.iterrows():
    reasons = []
    if r.ev_density > merge["ev_density"].median():
        reasons.append(f"EV density {r.ev_density:.0f} > median ({merge['ev_density'].median():.0f})")
    if r.population > merge["population"].median():
        reasons.append(f"population {int(r.population)} > median ({int(merge['population'].median())})")
    if r.overloaded:
        reasons.append(f"congestion index {r.congestion_index:.0f} — existing stations overloaded")
    if r.graph_recommended:
        reasons.append("Floyd-Warshall identified as optimal intermediate node")
    if r.existing_stations < 1:
        reasons.append("currently has NO charging station")
    if not reasons:
        reasons.append("moderate demand, adequately served — lower priority")
    print(f"\n  Rank {int(r.position)} — Zone {r.zone} (score={r.build_score:.4f})")
    for reason in reasons:
        print(f"    ▸ {reason}")
    print(f"    → Build {r.stations_recommended} new station(s) here")


SMART REROUTING: If Zone X is overloaded → go to Zone Y

  Zone B (OVERLOADED) → best reroute: Zone A
    Path       : B → A
    FW cost    : 12.954
    Dest loaded: No
  All options  :
    → A  path=B → A  eff_cost=12.954
    → E  path=B → A → E  eff_cost=112.097 [overloaded]
    → D  path=B → A → D  eff_cost=214.812
    → C  path=B → A → C  eff_cost=221.382

  Zone E (OVERLOADED) → best reroute: Zone A
    Path       : E → A
    FW cost    : 61.777
    Dest loaded: No
  All options  :
    → A  path=E → A  eff_cost=61.777
    → B  path=E → A → B  eff_cost=112.097 [overloaded]
    → D  path=E → D  eff_cost=196.958
    → C  path=E → C  eff_cost=208.26
EXPLAINABILITY SUMMARY — WHY EACH ZONE RANKED WHERE IT DID

  Rank 1 — Zone B (score=1.4678)
    ▸ EV density 400 > median (300)
    ▸ population 8063 > median (6044)
    ▸ congestion index 180388 — existing stations overloaded
    ▸ Floyd-Warshall identified as optimal intermediate node
    → Build 3 new station(s) here

  Rank 2 — Zone D

In [16]:
 
build_df.to_csv("station_build_priority.csv", index=False)
 

gap_export = []
for g in gap_results:
    gap_export.append({
        "gap_pair": g["gap_pair"],
        "raw_distance_km": g["raw_distance_km"],
        "direct_weight": g["direct_weight"],
        "recommended_zone": g["recommended_k"],
        "cost_reduction_pct": g["cost_reduction_%"],
        "benefit_score": g["benefit_score"],
        "fw_path": " → ".join(g["fw_path"]),
    })
pd.DataFrame(gap_export).to_csv("coverage_gap_analysis.csv", index=False)


In [17]:
reroute_export = []
for u in zones:
    if not G.nodes[u]["overloaded"]:
        continue
    alternatives = []
    for v in zones:
        if v == u: continue
        path = reconstruct_path(fw_pred, u, v)
        penalty = 1.5 if G.nodes[v]["overloaded"] else 1.0
        alternatives.append({
            "from_zone": u,
            "to_zone": v,
            "fw_cost": round(fw_dist[u][v], 3),
            "effective_cost": round(fw_dist[u][v] * penalty, 3),
            "path": " → ".join(path),
            "dest_overloaded": G.nodes[v]["overloaded"]
        })
    alternatives.sort(key=lambda x: x["effective_cost"])
    reroute_export.extend(alternatives)
pd.DataFrame(reroute_export).to_csv("smart_rerouting.csv", index=False)
 
print()

print("FILES SAVED")
print("  station_build_priority.csv")
print("  coverage_gap_analysis.csv")
print("smart_rerouting.csv")




FILES SAVED
  station_build_priority.csv
  coverage_gap_analysis.csv
smart_rerouting.csv
